# 5-Layer Autonomous GenAI-Powered Cloud Defense Architecture
## Viva Panel Resolution: Large-Scale Rigorous Pipeline

This notebook contains the fully optimized, dissertation-grade pipeline updated specifically to address the Viva Panel's ruthless review.

In [ ]:
# 1. Clone the GitHub Repository into the Colab Environment
!git clone https://github.com/tushars00d/GAICS_v2.git || echo "Already cloned"
%cd GAICS_v2

# 2. Install dependencies
!pip install -r requirements.txt -q

In [ ]:
import os
import shutil

# 3. Locate the dataset you uploaded and move it into the repository's data folder
possible_paths = [
    "/content/cicids_subset.csv",
    "/content/data/cicids_subset.csv",
    "/content/sample_data/cicids_subset.csv",
    "/content/sample_data/data/cicids_subset.csv"
]

target_path = "/content/GAICS_v2/data/cicids_subset.csv"
os.makedirs("/content/GAICS_v2/data", exist_ok=True)

found = False
if os.path.exists(target_path):
    print(f"Dataset already in correct location: {target_path}")
    found = True
else:
    for path in possible_paths:
        if os.path.exists(path):
            print(f"Found dataset at {path}. Moving to {target_path}...")
            shutil.move(path, target_path)
            found = True
            break

if not found:
    print("\n[!] ERROR: cicids_subset.csv not found!")
    print("Please check where you uploaded it in Colab and ensure the name is exactly 'cicids_subset.csv'")
else:
    print("\n[*] Dataset is ready. Proceeding with Pipeline.")

### Phase 1: High-Performance Generative Pre-training (Tabular DDPM)
Using Mixed Precision (AMP) to generate synthetic samples and augment the real CIC-IDS-2018 dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from training.train_ddpm import train_ddpm

sns.set_theme(style="whitegrid")

ddpm_model, ddpm_history = train_ddpm("configs/default.yaml")

plt.figure(figsize=(10, 4))
plt.plot(ddpm_history['loss'], label="DDPM MSE Loss", color='purple', linewidth=2)
plt.title("Tabular DDPM Generative Pre-training Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Phase 2: Robust Attention IDS Training
Training the detection model using Focal Loss and Cosine Schedulers.

In [ ]:
from training.train_ids import train_ids

ids_model, ids_history = train_ids("configs/default.yaml")

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Focal Loss', color='tab:red')
ax1.plot(ids_history['train_loss'], color='tab:red', label='Train Loss')
ax1.tick_params(axis='y', labelcolor='tab:red')

ax2 = ax1.twinx()
ax2.set_ylabel('Macro F1 Score', color='tab:blue')
ax2.plot(ids_history['val_f1'], color='tab:blue', label='Val Macro F1')
ax2.tick_params(axis='y', labelcolor='tab:blue')

plt.title("Attention IDS Training Curve")
fig.tight_layout()
plt.show()

### Phase 3: Explainability via SHAP (Addressing Viva Q8)
Generating quantitative SHAP values to explain which features the Attention-IDS focuses on.

In [ ]:
import shap
import torch
import numpy as np
from data.dataset_loaders import load_dataset
import yaml

with open("configs/default.yaml", "r") as f:
    config = yaml.safe_load(f)
    
_, test_loader, _, _ = load_dataset(config)
sample_data, _ = next(iter(test_loader))
sample_data = sample_data[:100].to(next(ids_model.parameters()).device)

explainer = shap.GradientExplainer(ids_model, sample_data)
shap_values = explainer.shap_values(sample_data)

shap.summary_plot(shap_values, sample_data.cpu().numpy(), feature_names=[f"F_{i}" for i in range(sample_data.shape[1])])

### Phase 4: Rigorous Ablation & Latency Profiling
Profiles end-to-end inference latency in milliseconds and calculates robust multi-seed Ablation Statistics.

In [ ]:
from evaluation.benchmark import run_ablation_studies
import IPython.display as display

results = run_ablation_studies("configs/default.yaml")

print("\nVisualizing Confusion Matrices from the final seed:\n")
display.display(display.Image(filename="plots/cm_clean.png"))
display.display(display.Image(filename="plots/cm_purified.png"))

### Phase 5: Autonomous SOAR execution & Bayesian Governance
Triggers the RAG-LLM Orchestrator to generate a response, which is then formally validated by the Bayesian PGM.

In [ ]:
from agents.agents import AutonomousResponseAgent
import json

agent = AutonomousResponseAgent(config)

# Simulating a high severity anomaly over critical infrastructure
decision = agent.process_incident(telemetry_vector=[0.5, 0.9, -0.2, 1.5], severity_score=0.9, asset_criticality=0.4)

print("\nFinal Execution Trace Object:\n", json.dumps(decision, indent=4))